# Active-fire ablation study

Answers Reviewer 1 ("provide ablation studies"), Reviewer 2 ("no numerical
table, repeated runs, or controlled comparison is reported" for the
spatial-only convolution claim) and Reviewer 3 ("a more systematic ablation
study is needed").

Four arms, three seeds each, with the epoch budget held fixed across all arms
so the comparison is controlled:

  full      - (1,3,3) convolutions, SE attention, deep supervision, dropout
  conv333   - full (3,3,3) convolutions      [the claim Reviewer 2 challenged]
  no_se     - SE blocks replaced by identity
  no_ds     - deep-supervision weight set to 0

Designed for Kaggle with 2x T4. Each job is one arm-seed pair. Jobs run two at
a time, one per GPU. Results are appended to a JSON after every job, and
completed jobs are skipped on restart, so the study can span several sessions
without losing work.

Budget: 12 jobs at roughly 3.3 h each, two in parallel = about 20 h total,
so 2-3 sessions. Reduce SEEDS to [42] for a single-seed pilot first.


In [1]:
import os, sys, glob, json, time, random, warnings, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from torch.amp import GradScaler, autocast
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")

try:
    import rasterio
except ImportError:
    os.system("pip install rasterio --quiet")
    import rasterio

N_GPU = torch.cuda.device_count()
print("PyTorch:", torch.__version__, "| GPUs:", N_GPU)
for i in range(N_GPU):
    print("  GPU {}: {}".format(i, torch.cuda.get_device_properties(i).name))

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)
RESULTS_JSON = os.path.join(OUT, "ablation_results.json")
CKPT_DIR = os.path.join(OUT, "ablation_ckpts")
os.makedirs(CKPT_DIR, exist_ok=True)


PyTorch: 2.10.0+cu128 | GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 1. Configuration

Identical to the main training run except MAX_EPOCHS, which is reduced and
held constant across arms so that no arm gets more optimisation than another.


In [2]:
class CFG:
    DATA_ROOT = ""
    for p in ["/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire/",
              "/kaggle/input/ts-satfire/ts-satfire/",
              "/kaggle/input/ts-satfire/ts-satfire/ts-satfire/",
              "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire/ts-satfire/"]:
        if os.path.exists(p):
            DATA_ROOT = p
            break

    TS_LENGTH = 2
    TRAIN_INTERVAL = 1
    IMAGE_SIZE = 256
    N_CHANNELS = 8

    MEAN = np.array([18.76488, 27.441864, 20.584806, 305.99478,
                     294.31738, 14.625097, 276.4207, 275.16766], dtype=np.float32)
    STD = np.array([15.911591, 14.879259, 10.832616, 21.761852,
                    24.703484, 9.878246, 40.64329, 40.7657], dtype=np.float32)

    MAX_EPOCHS = 40            # fixed budget for every arm
    BATCH_SIZE = 8
    LEARNING_RATE = 5e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    USE_AMP = True

    FOCAL_ALPHA = 0.75
    FOCAL_GAMMA = 2.0
    DICE_WEIGHT = 0.5
    FOCAL_WEIGHT = 0.5
    DS_WEIGHT = 0.3

    ENCODER_CHANNELS = [64, 128, 256, 512]
    DROPOUT = 0.1
    SE_REDUCTION = 8

    MIN_FIRE_PX = 10
    MAX_NEG_RATIO = 2
    SEED = 42

    VAL_IDS = ["20568194", "20701026", "20562846", "20700973", "24462610",
               "24462788", "24462753", "24103571", "21998313", "21751303",
               "22141596", "21999381", "22712904"]

    NO_LABEL_IDS = [
        "20777207", "20777386", "21693566", "21751309",
        "21889672", "21889683", "21889697", "21889719",
        "21889734", "21889754", "21997775", "22712973",
        "22713339", "23860939", "23860978", "23861018",
        "23861131", "24332700", "22712904",
    ]


assert CFG.DATA_ROOT, "TS-SatFire dataset not found"
print("DATA_ROOT:", CFG.DATA_ROOT)

ARMS = ["no_se", "no_ds", "dirty"]
SEEDS = [42]
JOBS = [(a, s) for a in ARMS for s in SEEDS]
print("Jobs:", len(JOBS))


DATA_ROOT: /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire/
Jobs: 3


## 2. Data pipeline

Copied from the main AF training notebook without modification.


In [3]:
def load_frame(fire_dir, day_path, return_label=False):
    with rasterio.open(day_path) as src:
        day_arr = src.read().astype(np.float32)
    day_bands = day_arr[:6]
    label = None
    if return_label and day_arr.shape[0] >= 7:
        b7 = day_arr[6]
        if np.isnan(b7).sum() < b7.size:
            label = (b7 >= 7).astype(np.float32)

    night_path = os.path.join(fire_dir, "VIIRS_Night",
        os.path.basename(day_path).replace("_VIIRS_Day", "_VIIRS_Night"))
    if os.path.exists(night_path):
        with rasterio.open(night_path) as src:
            na = src.read().astype(np.float32)
        nb = na[:2] if na.shape[0] >= 2 else np.zeros((2, *day_bands.shape[1:]), np.float32)
    else:
        nb = np.zeros((2, *day_bands.shape[1:]), np.float32)
    frame = np.concatenate([day_bands, nb], axis=0)
    return (frame, label) if return_label else frame


def check_day_has_label(day_path):
    with rasterio.open(day_path) as src:
        if src.count < 7:
            return False
        b7 = src.read(7).astype(np.float32)
        return np.isnan(b7).sum() < b7.size


class AFDatasetClean(Dataset):
    def __init__(self, fire_dirs, time_steps, interval, patch_size,
                 means, stds, augment=False, min_fire_px=10, max_neg_ratio=2,
                 seed=42, quiet=False):
        self.T, self.ps = time_steps, patch_size
        self.means, self.stds = means, stds
        self.augment = augment
        self.samples = []
        self._build_index(fire_dirs, interval, min_fire_px, max_neg_ratio, seed, quiet)

    def _build_index(self, fire_dirs, interval, min_fire_px, max_neg_ratio, seed, quiet):
        rng = random.Random(seed)
        for fd in fire_dirs:
            day_files = sorted(glob.glob(os.path.join(fd, "VIIRS_Day", "*.tif")))
            if len(day_files) < self.T:
                continue
            try:
                with rasterio.open(day_files[0]) as src:
                    if src.count < 7:
                        continue
                    H, W = src.height, src.width
            except Exception:
                continue
            if H < self.ps or W < self.ps:
                continue

            start = 0
            while start + self.T <= len(day_files):
                last_day = day_files[start + self.T - 1]
                if not check_day_has_label(last_day):
                    start += interval; continue
                lbl = None
                try:
                    with rasterio.open(last_day) as src:
                        if src.count >= 7:
                            b7 = src.read(7).astype(np.float32)
                            if np.isnan(b7).sum() < b7.size:
                                lbl = (b7 >= 7).astype(np.float32)
                except Exception:
                    pass
                if lbl is None:
                    start += interval; continue

                r0, c0 = (H - self.ps) // 2, (W - self.ps) // 2
                fire_px = int(lbl[r0:r0 + self.ps, c0:c0 + self.ps].sum())
                keep = (fire_px >= min_fire_px) or (rng.random() < 1.0 / (max_neg_ratio + 1))
                if keep:
                    self.samples.append({"fd": fd, "files": day_files,
                                         "start": start, "H": H, "W": W})
                start += interval
        if not quiet:
            print("  index built: {} samples".format(len(self.samples)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fd, H, W = s["fd"], s["H"], s["W"]
        win = s["files"][s["start"]:s["start"] + self.T]

        frames, label = [], None
        for t, dp in enumerate(win):
            if t == len(win) - 1:
                fr, label = load_frame(fd, dp, return_label=True)
            else:
                fr = load_frame(fd, dp)
            frames.append(fr[:, :H, :W])
        if label is None:
            label = np.zeros((H, W), np.float32)
        label = label[:H, :W]

        stack = np.stack(frames, axis=0)
        stack = (stack - self.means[None, :, None, None]) / (self.stds[None, :, None, None] + 1e-8)
        stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0)

        r0, c0 = (H - self.ps) // 2, (W - self.ps) // 2
        stack = stack[:, :, r0:r0 + self.ps, c0:c0 + self.ps]
        label = label[r0:r0 + self.ps, c0:c0 + self.ps]

        if self.augment:
            if random.random() > 0.5:
                stack = np.flip(stack, -1).copy(); label = np.flip(label, -1).copy()
            if random.random() > 0.5:
                stack = np.flip(stack, -2).copy(); label = np.flip(label, -2).copy()
            k = random.randint(0, 3)
            if k:
                stack = np.rot90(stack, k, axes=(-2, -1)).copy()
                label = np.rot90(label, k, axes=(0, 1)).copy()

        x = torch.from_numpy(stack.transpose(1, 0, 2, 3).copy()).float()
        y = torch.from_numpy(label.copy()).long()
        return x, y


all_ids = sorted(os.listdir(CFG.DATA_ROOT))
numeric_ids = [d for d in all_ids if d.isdigit()]
train_ids = [d for d in numeric_ids if d not in CFG.VAL_IDS and d not in CFG.NO_LABEL_IDS]
dirty_ids = [d for d in numeric_ids if d not in CFG.VAL_IDS]   # keeps the mislabelled fires
DIRTY_FIRES = [os.path.join(CFG.DATA_ROOT, d) for d in dirty_ids]
print("dirty train fires:", len(DIRTY_FIRES), "(vs clean", len(train_ids), ")")
val_ids = [d for d in numeric_ids if d in CFG.VAL_IDS and d not in CFG.NO_LABEL_IDS]
TRAIN_FIRES = [os.path.join(CFG.DATA_ROOT, d) for d in train_ids]
VAL_FIRES = [os.path.join(CFG.DATA_ROOT, d) for d in val_ids]
print("train fires:", len(TRAIN_FIRES), "| val fires:", len(VAL_FIRES))

print("Building shared indices once (reused by every job)...")
TRAIN_DS = AFDatasetClean(TRAIN_FIRES, CFG.TS_LENGTH, CFG.TRAIN_INTERVAL, CFG.IMAGE_SIZE,
                          CFG.MEAN, CFG.STD, augment=True,
                          min_fire_px=CFG.MIN_FIRE_PX, max_neg_ratio=CFG.MAX_NEG_RATIO,
                          seed=CFG.SEED)
VAL_DS = AFDatasetClean(VAL_FIRES, CFG.TS_LENGTH, CFG.TRAIN_INTERVAL, CFG.IMAGE_SIZE,
                        CFG.MEAN, CFG.STD, augment=False,
                        min_fire_px=CFG.MIN_FIRE_PX, max_neg_ratio=CFG.MAX_NEG_RATIO,
                        seed=CFG.SEED)
DIRTY_DS = AFDatasetClean(DIRTY_FIRES, CFG.TS_LENGTH, CFG.TRAIN_INTERVAL, CFG.IMAGE_SIZE,
                          CFG.MEAN, CFG.STD, augment=True,
                          min_fire_px=CFG.MIN_FIRE_PX, max_neg_ratio=CFG.MAX_NEG_RATIO,
                          seed=CFG.SEED)


dirty train fires: 138 (vs clean 120 )
train fires: 120 | val fires: 12
Building shared indices once (reused by every job)...
  index built: 1719 samples
  index built: 203 samples
  index built: 1719 samples


## 3. Model, parameterised by ablation arm

`kernel` switches (1,3,3) against (3,3,3). `use_se` switches SE attention.
`ds_weight` at 0 disables deep supervision. Everything else is identical.


In [4]:
class SEBlock3D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(nn.Linear(ch, ch // r, bias=False), nn.ReLU(True),
                                nn.Linear(ch // r, ch, bias=False), nn.Sigmoid())

    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1, 1)


class ResBlock3D(nn.Module):
    def __init__(self, ic, oc, r=8, dr=0.1, kernel=(1, 3, 3), use_se=True):
        super().__init__()
        pad = tuple(k // 2 for k in kernel)
        self.c1 = nn.Conv3d(ic, oc, kernel, padding=pad, bias=False)
        self.b1 = nn.BatchNorm3d(oc)
        self.c2 = nn.Conv3d(oc, oc, kernel, padding=pad, bias=False)
        self.b2 = nn.BatchNorm3d(oc)
        self.se = SEBlock3D(oc, r) if use_se else nn.Identity()
        self.relu = nn.ReLU(True)
        self.drop = nn.Dropout3d(dr) if dr > 0 else nn.Identity()
        self.skip = (nn.Sequential(nn.Conv3d(ic, oc, 1, bias=False), nn.BatchNorm3d(oc))
                     if ic != oc else nn.Identity())

    def forward(self, x):
        r = self.skip(x)
        o = self.relu(self.b1(self.c1(x)))
        o = self.drop(o)
        o = self.se(self.b2(self.c2(o)))
        return self.relu(o + r)


class SEUNet3D(nn.Module):
    def __init__(self, ic=8, nc=1, ec=(64, 128, 256, 512), r=8, dr=0.1,
                 kernel=(1, 3, 3), use_se=True):
        super().__init__()
        kw = dict(r=r, dr=dr, kernel=kernel, use_se=use_se)
        self.e1 = ResBlock3D(ic, ec[0], **kw)
        self.e2 = ResBlock3D(ec[0], ec[1], **kw)
        self.e3 = ResBlock3D(ec[1], ec[2], **kw)
        self.e4 = ResBlock3D(ec[2], ec[3], **kw)
        self.pool = nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2))
        self.bot = ResBlock3D(ec[3], ec[3] * 2, **kw)
        self.u4 = nn.ConvTranspose3d(ec[3] * 2, ec[3], (1, 2, 2), stride=(1, 2, 2))
        self.d4 = ResBlock3D(ec[3] * 2, ec[3], **kw)
        self.u3 = nn.ConvTranspose3d(ec[3], ec[2], (1, 2, 2), stride=(1, 2, 2))
        self.d3 = ResBlock3D(ec[2] * 2, ec[2], **kw)
        self.u2 = nn.ConvTranspose3d(ec[2], ec[1], (1, 2, 2), stride=(1, 2, 2))
        self.d2 = ResBlock3D(ec[1] * 2, ec[1], **kw)
        self.u1 = nn.ConvTranspose3d(ec[1], ec[0], (1, 2, 2), stride=(1, 2, 2))
        self.d1 = ResBlock3D(ec[0] * 2, ec[0], **kw)
        self.final = nn.Conv3d(ec[0], nc, 1)
        self.ds3 = nn.Conv3d(ec[2], nc, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        b = self.bot(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.final(d1), self.ds3(d3)


def build_model(arm):
    kernel = (3, 3, 3) if arm == "conv333" else (1, 3, 3)
    use_se = (arm != "no_se")
    return SEUNet3D(ic=CFG.N_CHANNELS, ec=tuple(CFG.ENCODER_CHANNELS),
                    r=CFG.SE_REDUCTION, dr=CFG.DROPOUT,
                    kernel=kernel, use_se=use_se)


class DiceFocalLoss(nn.Module):
    def __init__(self, dw=0.5, fw=0.5, gamma=2.0, alpha=0.75, dsw=0.3):
        super().__init__()
        self.dw, self.fw, self.gamma, self.alpha, self.dsw = dw, fw, gamma, alpha, dsw

    def _dice(self, p, t):
        ps = torch.sigmoid(p).reshape(-1); tf = t.reshape(-1)
        return 1 - (2 * (ps * tf).sum() + 1) / (ps.sum() + tf.sum() + 1)

    def _focal(self, p, t):
        bce = F.binary_cross_entropy_with_logits(p, t, reduction="none")
        pt = torch.sigmoid(p) * t + (1 - torch.sigmoid(p)) * (1 - t)
        at = self.alpha * t + (1 - self.alpha) * (1 - t)
        return (at * (1 - pt) ** self.gamma * bce).mean()

    def _loss(self, p, t):
        return self.dw * self._dice(p, t) + self.fw * self._focal(p, t)

    def forward(self, main, ds, target):
        loss = self._loss(main, target)
        if self.dsw > 0 and ds is not None:
            t_small = F.interpolate(target.unsqueeze(1), size=ds.shape[-2:],
                                    mode="nearest").squeeze(1)
            loss = loss + self.dsw * self._loss(ds, t_small)
        return loss


## 4. One training job


In [5]:
def run_job(arm, seed, device):
    tag = "{}_s{}".format(arm, seed)
    ck_path = os.path.join(CKPT_DIR, tag + ".pt")

    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = build_model(arm).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    ds = DIRTY_DS if arm == "dirty" else TRAIN_DS
    train_loader = DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(VAL_DS, batch_size=CFG.BATCH_SIZE, shuffle=False,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True)

    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE,
                            weight_decay=CFG.WEIGHT_DECAY)
    sched = OneCycleLR(opt, max_lr=CFG.LEARNING_RATE, steps_per_epoch=len(train_loader),
                       epochs=CFG.MAX_EPOCHS, pct_start=0.1, anneal_strategy="cos")
    scaler = GradScaler("cuda", enabled=CFG.USE_AMP)
    ds_w = 0.0 if arm == "no_ds" else CFG.DS_WEIGHT
    crit = DiceFocalLoss(CFG.DICE_WEIGHT, CFG.FOCAL_WEIGHT,
                         CFG.FOCAL_GAMMA, CFG.FOCAL_ALPHA, ds_w)

    start_ep, best_f1, best_iou, best_ep, history = 0, 0.0, 0.0, 0, []
    if os.path.exists(ck_path):
        ck = torch.load(ck_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        start_ep = ck["epoch"] + 1
        best_f1, best_iou, best_ep = ck["best_f1"], ck["best_iou"], ck["best_ep"]
        history = ck["history"]
        print("[{}] resuming at epoch {}".format(tag, start_ep), flush=True)

    t0 = time.time()
    for ep in range(start_ep, CFG.MAX_EPOCHS):
        model.train()
        tot = 0.0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).float()
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=CFG.USE_AMP):
                main, ds = model(x)
                loss = crit(main[:, 0, -1], ds[:, 0, -1], y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()

        model.eval()
        tp = fp = fn = 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True).float()
                with autocast("cuda", enabled=CFG.USE_AMP):
                    main, _ = model(x)
                p = (torch.sigmoid(main[:, 0, -1].float()) > 0.25)
                t = y > 0.5
                tp += int((p & t).sum()); fp += int((p & ~t).sum()); fn += int((~p & t).sum())
        f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
        iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
        history.append({"epoch": ep, "train_loss": tot / max(len(train_loader), 1),
                        "val_f1": f1, "val_iou": iou})
        if f1 > best_f1:
            best_f1, best_iou, best_ep = f1, iou, ep

        torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": ep, "best_f1": best_f1, "best_iou": best_iou,
                    "best_ep": best_ep, "history": history}, ck_path)
        if (ep + 1) % 5 == 0 or ep == CFG.MAX_EPOCHS - 1:
            print("[{}] ep {}/{}  val F1 {:.4f}  best {:.4f}".format(
                tag, ep + 1, CFG.MAX_EPOCHS, f1, best_f1), flush=True)

    return {"arm": arm, "seed": seed, "n_params": n_params,
            "best_val_f1": best_f1, "best_val_iou": best_iou,
            "best_epoch": best_ep, "epochs": CFG.MAX_EPOCHS,
            "hours": (time.time() - t0) / 3600.0, "history": history}


## 5. Run the study

Completed jobs are skipped, so rerunning the notebook in a new session
continues where the previous one stopped.


In [6]:
results = {}
if os.path.exists(RESULTS_JSON):
    results = json.load(open(RESULTS_JSON))
    print("Already complete:", sorted(results.keys()))

pending = [(a, s) for a, s in JOBS if "{}_s{}".format(a, s) not in results]
print("Pending jobs:", len(pending))

lock = __import__("threading").Lock()


def worker(rank):
    device = torch.device("cuda", rank if N_GPU > 0 else 0)
    for a, s in pending[rank::max(N_GPU, 1)]:
        tag = "{}_s{}".format(a, s)
        print("[gpu {}] starting {}".format(rank, tag), flush=True)
        try:
            r = run_job(a, s, device)
        except Exception as e:
            r = {"arm": a, "seed": s, "error": "{}: {}".format(type(e).__name__, e)}
            print("[gpu {}] FAILED {}: {}".format(rank, tag, e), flush=True)
        with lock:
            results[tag] = r
            json.dump(results, open(RESULTS_JSON, "w"), indent=1)
        print("[gpu {}] finished {}".format(rank, tag), flush=True)


if pending:
    n = max(N_GPU, 1)
    if n > 1:
        with ThreadPoolExecutor(max_workers=n) as ex:
            futs = [ex.submit(worker, r) for r in range(n)]
            for f in futs:
                f.result()
    else:
        worker(0)
else:
    print("Nothing pending.")


Pending jobs: 3
[gpu 0] starting no_se_s42
[gpu 1] starting no_ds_s42
[no_se_s42] ep 5/40  val F1 0.7735  best 0.7962
[no_ds_s42] ep 5/40  val F1 0.7831  best 0.7831
[no_se_s42] ep 10/40  val F1 0.4961  best 0.8040
[no_ds_s42] ep 10/40  val F1 0.8095  best 0.8095
[no_se_s42] ep 15/40  val F1 0.8157  best 0.8157
[no_ds_s42] ep 15/40  val F1 0.8097  best 0.8134
[no_se_s42] ep 20/40  val F1 0.8032  best 0.8157
[no_ds_s42] ep 20/40  val F1 0.8183  best 0.8183
[no_se_s42] ep 25/40  val F1 0.8066  best 0.8157
[no_ds_s42] ep 25/40  val F1 0.8169  best 0.8207
[no_se_s42] ep 30/40  val F1 0.8105  best 0.8219
[no_ds_s42] ep 30/40  val F1 0.8162  best 0.8207
[no_se_s42] ep 35/40  val F1 0.8215  best 0.8219
[no_ds_s42] ep 35/40  val F1 0.8144  best 0.8228
[no_se_s42] ep 40/40  val F1 0.8210  best 0.8219
[gpu 0] finished no_se_s42
[gpu 0] starting dirty_s42
[no_ds_s42] ep 40/40  val F1 0.8221  best 0.8229
[gpu 1] finished no_ds_s42
[dirty_s42] ep 5/40  val F1 0.7890  best 0.7890
[dirty_s42] ep 10/4

## 6. Ablation table


In [7]:
import pandas as pd

rows = []
for tag, r in results.items():
    if "error" in r:
        rows.append({"arm": r["arm"], "seed": r["seed"], "val_f1": np.nan,
                     "val_iou": np.nan, "params_M": np.nan, "error": r["error"]})
        continue
    rows.append({"arm": r["arm"], "seed": r["seed"], "val_f1": r["best_val_f1"],
                 "val_iou": r["best_val_iou"], "params_M": r["n_params"] / 1e6,
                 "best_epoch": r["best_epoch"], "hours": r.get("hours", np.nan),
                 "error": None})
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT, "ablation_per_run.csv"), index=False)

if df.val_f1.notna().any():
    agg = (df.dropna(subset=["val_f1"])
             .groupby("arm")
             .agg(n=("seed", "count"),
                  f1_mean=("val_f1", "mean"), f1_std=("val_f1", "std"),
                  iou_mean=("val_iou", "mean"), iou_std=("val_iou", "std"),
                  params_M=("params_M", "first"))
             .reindex(ARMS).dropna(how="all"))
    agg.to_csv(os.path.join(OUT, "ablation_summary.csv"))

    print("\n" + "=" * 74)
    print("ABLATION SUMMARY  (validation, {} epochs fixed per arm)".format(CFG.MAX_EPOCHS))
    print("=" * 74)
    print("{:<10} {:>3} {:>16} {:>16} {:>10}".format(
        "arm", "n", "val F1", "val IoU", "params M"))
    print("-" * 74)
    for arm in ARMS:
        if arm not in agg.index:
            continue
        r = agg.loc[arm]
        sd_f1 = 0.0 if pd.isna(r.f1_std) else r.f1_std
        sd_iou = 0.0 if pd.isna(r.iou_std) else r.iou_std
        print("{:<10} {:>3d} {:>9.4f} +/- {:.4f} {:>9.4f} +/- {:.4f} {:>10.2f}".format(
            arm, int(r.n), r.f1_mean, sd_f1, r.iou_mean, sd_iou, r.params_M))
    print("-" * 74)

    if "full" in agg.index:
        base = agg.loc["full", "f1_mean"]
        print("\nChange relative to the full model:")
        for arm in ARMS:
            if arm == "full" or arm not in agg.index:
                continue
            print("  {:<10} {:+.4f}".format(arm, agg.loc[arm, "f1_mean"] - base))
        print("\nThe conv333 row is the controlled comparison Reviewer 2 asked for.")

print("\nFiles written:")
for f in sorted(glob.glob(os.path.join(OUT, "ablation*.csv")) + [RESULTS_JSON]):
    if os.path.exists(f):
        print("  {:>8.1f} KB  {}".format(os.path.getsize(f) / 1e3, f))



ABLATION SUMMARY  (validation, 40 epochs fixed per arm)
arm          n           val F1          val IoU   params M
--------------------------------------------------------------------------
no_se        1    0.8219 +/- 0.0000    0.6976 +/- 0.0000      32.44
no_ds        1    0.8229 +/- 0.0000    0.6991 +/- 0.0000      32.88
dirty        1    0.8208 +/- 0.0000    0.6961 +/- 0.0000      32.88
--------------------------------------------------------------------------

Files written:
       0.3 KB  /kaggle/working/ablation_per_run.csv
      16.8 KB  /kaggle/working/ablation_results.json
       0.2 KB  /kaggle/working/ablation_summary.csv
